In [1]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm
import csv
from scipy.sparse import issparse
from model.sparsemax import Sparsemax


from model.model import MIL, EarlyStopping
from model.dataset import BagsDataset, custom_collate_fn

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
#adata_1 = sc.read_h5ad('../test.h5ad')
#adata_2 = sc.read_h5ad('/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE/HumanLungCancerFFPE.h5ad')


In [4]:
"""adata_radius_input = [
    (adata_1, 200, 'low'),
    (adata_2, 200, 'low'),
]"""

"adata_radius_input = [\n    (adata_1, 200, 'low'),\n    (adata_2, 200, 'low'),\n]"

In [97]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/VisiumHD/HumanLungCancer/T_cell.h5ad')

In [99]:
dataset = BagsDataset(adata, immune_cell='tcell', radius=75, n_genes=500, resolution='high')
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
#dataset = load_datasets(data_dir='/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE', immune_cell='tcell',radius=200,)

Tumor cells shape after filtering: (71423, 18085)
Selecting top 500 genes based on mean expression
Preprocessed data: (150497, 500)


Creating Bags with radius 75: 100%|███████████████████████| 150497/150497 [00:41<00:00, 3665.64it/s]


Total bags created: 72409
Average instances per bag: 46


In [70]:
dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)



In [ ]:
distances, gene_expressions, labels, core_index, current_genes = next(iter(dataloader))


In [71]:
class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        self.sparsemax = Sparsemax(dim=0)
    
    def forward(self, x):
        #print(x)
        a = self.a
        x = self.sparsemax(-torch.exp(a) * x)
        return x

In [72]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[0.],
        [0.],
        [0.],
        [1.]], grad_fn=<TransposeBackward0>)


In [73]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        b = self.b
        x = self.softmax(torch.exp(b) * x)
        return x

In [74]:
gene_expressions[0]

tensor([[5.3232, 5.0212, 4.8466,  ..., 1.7678, 0.0000, 0.0000],
        [5.1958, 4.9436, 5.1062,  ..., 1.4098, 1.4098, 1.4098],
        [4.8444, 4.3001, 4.3942,  ..., 1.7660, 1.7660, 2.1128],
        [5.4735, 5.1787, 5.0111,  ..., 0.0000, 1.3968, 1.9579]])

In [75]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output)

tensor([[2.7145e-01, 1.1946e-01, 7.4314e-02,  ..., 1.7236e-05, 1.4109e-07,
         1.4109e-07],
        [1.9822e-01, 9.9869e-02, 1.5536e-01,  ..., 6.7238e-06, 6.7238e-06,
         6.7238e-06],
        [2.5712e-01, 5.8557e-02, 7.5621e-02,  ..., 5.9693e-05, 5.9693e-05,
         1.5321e-04],
        [3.5704e-01, 1.6020e-01, 1.0158e-01,  ..., 1.2332e-07, 5.4956e-06,
         2.5259e-05]], grad_fn=<SoftmaxBackward0>)


In [76]:
distances, gene_expressions, labels, core_index, current_genes = next(iter(dataloader))

In [77]:
class Immunogenicity(nn.Module):
    def __init__(self, all_genes):
        super(Immunogenicity, self).__init__()
        self.all_genes = all_genes
        self.gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}
        self.ig = nn.Parameter(torch.full((len(all_genes),), -1.0), requires_grad=True)
    
    def forward(self, current_genes):
        # Filter genes to include only those present in all_genes
        filtered_genes = [gene for gene in current_genes if gene in self.gene_to_index]
        indices = [self.gene_to_index[gene] for gene in filtered_genes]
        ig = torch.sigmoid(self.ig[indices])
        return ig, filtered_genes

In [78]:
all_genes = pd.read_csv('./data/tumor_antigens.csv')
all_genes = all_genes['Gene'].values.tolist()

In [79]:
model = Immunogenicity(all_genes)

In [80]:
output, filtered_genes = model(list(current_genes[0]))

In [81]:
len(output)

10

In [82]:
class MIL(nn.Module):
    def __init__(self, all_genes):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.immunogenicity = Immunogenicity(all_genes)
        self.alpha = nn.Parameter(torch.tensor(1.0), requires_grad=True)
        self.beta = nn.Parameter(torch.tensor(1.0), requires_grad=True)
    
    def forward(self, distances, gene_expressions, current_genes):
        bag_outputs = []
        for distance, gene_expression in zip(distances, gene_expressions):
            distance = self.distance(distance)
            gene_expression = self.gene_expression(gene_expression)
            immunogenicity, filtered_genes = self.immunogenicity(current_genes)
            alpha = self.alpha
            beta = self.beta
        
            if len(filtered_genes) == 0:
                continue  # Skip if no overlapping genes
        
        # Print debug information
            #print(f"gene_expression shape: {gene_expression.shape}")
            #print(f"current_genes length: {len(current_genes)}")
            #print(f"filtered_genes length: {len(filtered_genes)}")
        
        
            gene_to_index = {gene: idx for idx, gene in enumerate(current_genes)}
        
            gene_indices = [gene_to_index[gene] for gene in filtered_genes if gene in gene_to_index]
            gene_expression = gene_expression[:, gene_indices]
        
            #print(f"Filtered gene_expression shape: {gene_expression.shape}")
            #print(f"Immunogenicity shape: {immunogenicity.shape}")
        
            z = gene_expression @ immunogenicity
            #print(f"z shape: {z.shape}")
            z = z.unsqueeze(1)
            #print(f"z shape: {z.shape}")
            #print(f"distance shape: {distance}")
            bag_output = distance * z
            bag_output = torch.sum(bag_output, dim=0)
            bag_output = torch.exp(alpha) * bag_output + beta
            bag_output = torch.sigmoid(bag_output)
            #print(bag_output)
            bag_outputs.append(bag_output)
            #df = pd.DataFrame(bag_outputs)
            #df.to_csv('output.csv')
    
        
        return torch.stack(bag_outputs).squeeze(dim=1)

In [83]:
model = MIL(all_genes)

In [84]:
output = model(distances, gene_expressions, list(current_genes[0]))
output

tensor([0.7311], grad_fn=<SqueezeBackward1>)

In [85]:
labels[0]


tensor(1.)

In [86]:
make_dot(output, params=dict(model.named_parameters())).render("MIL_computational_graph", format="png")


'MIL_computational_graph.png'

In [87]:
model = MIL(all_genes)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
model.to(device)

MIL(
  (distance): Distance(
    (sparsemax): Sparsemax()
  )
  (gene_expression): Gene_expression(
    (softmax): Softmax(dim=-1)
  )
  (immunogenicity): Immunogenicity()
)

In [88]:
!pwd

/project/DPDS/Wang_lab/s439765/spatial_tcr/MIL_TCR


In [89]:
ig_scores_before_training = model.immunogenicity.ig
with open('ig_scores_before_training.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score Before Training'])
    for gene, score in zip(all_genes, ig_scores_before_training):
        writer.writerow([gene, score.item()])

In [90]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = np.inf
        self.counter = 0
        self.stopped_epoch = 0
        self.early_stop = False

    def __call__(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), 'final_model.pth')
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch

In [95]:

model = MIL(all_genes).to(device)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
num_epochs = 100
patience = 5
early_stopping = EarlyStopping(patience=patience, delta=0.1)

In [96]:
for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, current_genes) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()

            distances = torch.stack(distances).to(device)
            gene_expressions = torch.stack(gene_expressions).to(device)
            label = label.clone().detach().float().to(device)
            
            output = model(distances, gene_expressions, list(current_genes[0]))
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_distances, val_gene_expressions, val_label, _, val_current_genes in val_dataloader:
            val_distances = torch.stack(val_distances).to(device)
            val_gene_expressions = torch.stack(val_gene_expressions).to(device)
            val_label = val_label.clone().detach().float().to(device)
            val_output = model(val_distances, val_gene_expressions, list(val_current_genes[0]))
            val_loss += criterion(val_output, val_label).item()
    
    val_loss /= len(val_dataloader)
    print(f'Validation Loss: {val_loss:.4f}')

    """early_stopping(val_loss, model, epoch)
    if early_stopping.early_stop:
        print(f'Early stopping at epoch {epoch+1}')
        break"""
        

Epoch 1/100: 100%|██████████| 858/858 [00:01<00:00, 477.66batch/s, loss=1.19] 


Epoch [1/100], Loss: 0.7921
Validation Loss: 0.7771


Epoch 2/100: 100%|██████████| 858/858 [00:01<00:00, 444.83batch/s, loss=1.08] 


Epoch [2/100], Loss: 0.7599
Validation Loss: 0.7499


Epoch 3/100: 100%|██████████| 858/858 [00:01<00:00, 488.16batch/s, loss=0.458]


...

Epoch [55/100], Loss: 0.6933
Validation Loss: 0.6932


Epoch 56/100: 100%|██████████| 858/858 [00:01<00:00, 451.63batch/s, loss=0.696]


Epoch [56/100], Loss: 0.6933
Validation Loss: 0.6932


Epoch 57/100:   7%|▋         | 61/858 [00:00<00:01, 448.63batch/s, loss=0.696]


KeyboardInterrupt: 

In [48]:
def predict(model, adata, device, radius=200, max_instances=None):
    model.eval()
    
    dataset = BagsDataset(adata, immune_cell='tcell', radius=radius, max_instances=max_instances)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=custom_collate_fn)
    
    predictions = np.full(len(adata.obs), np.nan)  # Initialize predictions array with NaNs

    with torch.no_grad():
        with tqdm(dataloader, unit="batch") as tepoch:
            for distances, gene_expressions, _, core_idx, current_genes in tepoch:
                tepoch.set_description("Predicting")
                
                # Move data to the device
                distances = [d.to(device) for d in distances]
                gene_expressions = [g.to(device) for g in gene_expressions]
                
                output = model(distances, gene_expressions, list(current_genes[0]))
                
                # Ensure we extract a single element from core_idx and output before using them
                predictions[int(core_idx.item())] = output.cpu().numpy().flatten().item()

    adata.obs['tcr_predict'] = predictions
    return adata

In [49]:
adata = sc.read_h5ad('../test.h5ad')

In [50]:
adata = predict(model, adata,radius=200, device=device)

Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:00<00:00, 6262.95it/s]


Total bags created: 3858
Average instances per bag: 9


Predicting: 100%|██████████| 3858/3858 [00:04<00:00, 846.80batch/s]


In [51]:
adata.obs

,X,Y,T,cell_type,tcr_predict
AACACGTGCATCGCAC-1,7600,2200,0,0,0.505103
AACACTTGGCAAGGAA-1,4700,7100,1,1,0.557644
AACAGGATTCATAGTT-1,4900,4300,1,0,0.717114
AACAGGTTATTGCACC-1,2800,8600,0,0,0.500146
AACAGGTTCACCGAAG-1,5100,4100,1,0,0.500000
...,...,...,...,...,...
TGTTGGAACCTTCCGC-1,3500,3500,0,0,0.516184
TGTTGGAACGAGGTCA-1,2800,7200,0,1,0.544874
TGTTGGAAGCTCGGTA-1,100,9500,0,0,0.721982
TGTTGGATGGACTTCT-1,1300,5300,0,0,0.512577


In [46]:
print(torch.sigmoid(model.immunogenicity.ig))

tensor([0.2342, 0.2457, 0.2458,  ..., 0.2689, 0.2689, 0.2689],
       grad_fn=<SigmoidBackward0>)


In [44]:
ig_scores_after_training = model.immunogenicity.ig
with open('ig_scores_after_training.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score After Training'])
    for gene, score in zip(all_genes, ig_scores_after_training):
        writer.writerow([gene, score.item()])
        
with open('ig_score_changes.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score Before Training', 'IG Score After Training'])
    for gene, before, after in zip(all_genes, ig_scores_before_training, ig_scores_after_training):
        writer.writerow([gene, before.item(), after.item()])